# weight_ttest – Summary

This notebook applies **independent-sample t-tests** to assess whether boat weight differences between runs have a statistically significant impact on performance metrics, primarily **SOG** (Speed Over Ground).  
The analysis uses telemetry from `all_data.csv` filtered for **June 9, 2025 runs**.

---

## Inputs
- **Data**: `all_data.csv` containing time-series telemetry across runs.  
- **Helper function**:  
  - `t_test(df1, df2, target="SOG")`: runs a two-sample t-test on the specified column, prints t-statistic, p-value, and interprets significance (`p < 0.05`).

---

## Workflow

### Step 1: Load & filter data
- Restrict dataset to rows with timestamp starting `"2025-06-09"`.  

### Step 2: Define groups of runs
- **First runs (5,6,7)**: Karl holds weights.  
- **Last runs (8–11)**: Gian holds 6 kg.  
- Subsets created: `data_9juin_first_runs`, `data_9juin_last_runs`.

### Step 3: Initial t-test (wind conditions)
- Compare **TWS** (True Wind Speed) between early (5–7) and later (8–11) runs to verify comparable conditions.

### Step 4: Karl heavy vs Karl light
- Subset only Karl’s boat (or SenseBoard when opponent is Gian).  
- Run t-tests on **SOG**:
  - General (all legs).  
  - Upwind only (`TWA > 0`).  
  - Downwind only (`TWA ≤ 0`).  
- For each case: report boat weight, mean SOG, std SOG.

### Step 5: Gian light vs Gian heavy
- Subset only Gian’s boat (or SenseBoard when opponent is Karl).  
- Run t-tests on **SOG**:
  - General.  
  - Upwind only.  
  - Downwind only.  
- For each case: report boat weight, mean SOG, std SOG.

---

## Output
- Printed results for each t-test: **t-statistic**, **p-value**, interpretation of significance.  
- Descriptive stats: mean and std of `SOG`, average `boat_weight` for each group.  
- Group comparisons structured as:  
  - **Karl**: heavy (runs 5–7) vs light (runs 8–11).  
  - **Gian**: light (runs 5–7) vs heavy (runs 8–11).  
  - Separate breakdowns for **upwind** and **downwind**.

---

## Notes
- Significance threshold: `p < 0.05`.  
- If not significant, conclusion is to **combine data**; if significant, keep data split by weight condition.  


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import scipy.stats as stats

def t_test(df1, df2, target="SOG"):
    t_stat, p_value = stats.ttest_ind(df1[target].dropna(), df2[target].dropna())
    print(f"T-statistic: {t_stat:.3f}, p-value: {p_value:.15f}")
    
    # If p-value is less than 0.05, the difference is statistically significant
    if p_value < 0.05:
        print("The difference is statistically significant, keeping data split.")
    else:
        print("The difference is not statistically significant, keeping data combined.")


In [2]:
df = pd.read_csv("all_data.csv")

In [3]:
data_9juin = df[df["ISODateTimeUTC"].str.startswith("2025-06-09")]

## T test on the TWS between runs 5,6,7 where Karl holds the weights and runs 8,9,10,11 where Gian holds 6kgs

In [4]:
first_runs = ["09_06_2025_Run5","09_06_2025_Run6","09_06_2025_Run7"]
data_9juin_first_runs = data_9juin[data_9juin["run"].isin(first_runs) ]

In [5]:
last_runs = ["09_06_2025_Run8","09_06_2025_Run9","09_06_2025_Run10","09_06_2025_Run11"]
data_9juin_last_runs = data_9juin[data_9juin["run"].isin(last_runs) ]

In [6]:
t_test(data_9juin_first_runs,data_9juin_last_runs, target="TWS")
print(data_9juin_first_runs["TWS"].mean(),data_9juin_last_runs["TWS"].mean())
print(f"Average TWS in Group 1: {data_9juin_first_runs['TWS'].mean()},STD TWS in Group 1: {data_9juin_first_runs['TWS'].std()}")
print(f"Average TWS in Group 2: {data_9juin_last_runs['TWS'].mean()},STD TWS in Group 2: {data_9juin_last_runs['TWS'].std()}")

T-statistic: -20.004, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.
7.2590247128437175 7.541569498486814
Average TWS in Group 1: 7.2590247128437175,STD TWS in Group 1: 0.909090314209303
Average TWS in Group 2: 7.541569498486814,STD TWS in Group 2: 0.7956380127932735


## t test karl heavy vs karl not heavy

In [7]:
only_karl_first_runs_heavy = data_9juin_first_runs[
    (data_9juin_first_runs["boat_name"] == "Karl Maeder") |
    ((data_9juin_first_runs["boat_name"] == "SenseBoard") & 
     (data_9juin_first_runs["opponent_name"] == "Gian Stragiotti"))
]
only_karl_first_runs_heavy.sample(5)

,ISODateTimeUTC,SecondsSince1970,Heel_Abs,Heel_Lwd,Lat,LatBow,LatCenter,LatStern,Leg,Line_C,...,interval_duration,mast_brand,gain_forward,gain_lateral,gain_vmg,Line_R2,Line_L2,Line_C2,side_line2,total_line2
47087,2025-06-09T12:54:11.760Z,1.749474e+09,56.4,56.4,43.507859,43.507857,43.507863,43.507869,NaN,116.8,...,68.880,Levi,1.584206,0.477543,1.405222,6.6,7.393,116.8,13.993,130.793
45380,2025-06-09T12:46:59.653Z,1.749473e+09,57.1,57.1,43.501582,43.501580,43.501586,43.501592,1.0,105.4,...,68.000,Levi,17.912566,4.148307,15.144641,8.9,10.000,105.4,18.900,124.300
47682,2025-06-09T12:55:11.253Z,1.749474e+09,52.0,52.0,43.501864,43.501862,43.501869,43.501875,NaN,142.5,...,68.880,Levi,-0.768628,-17.263470,-13.552594,6.4,8.300,142.5,14.700,157.200
46500,2025-06-09T12:49:18.063Z,1.749473e+09,44.1,44.1,43.507428,43.507430,43.507424,43.507418,1.0,92.7,...,47.292,Levi,8.932596,-2.697973,-8.416439,4.9,6.396,92.7,11.296,103.996
43394,2025-06-09T12:37:50.558Z,1.749473e+09,53.9,53.9,43.507896,43.507895,43.507901,43.507907,NaN,112.8,...,68.713,Levi,0.852267,-0.078085,0.483122,2.9,6.100,112.8,9.000,121.800


In [8]:
only_karl_last_runs_light = data_9juin_last_runs[
    (data_9juin_last_runs["boat_name"] == "Karl Maeder") |
    ((data_9juin_last_runs["boat_name"] == "SenseBoard") & 
     (data_9juin_last_runs["opponent_name"] == "Gian Stragiotti"))
]
only_karl_last_runs_light.sample(5)

,ISODateTimeUTC,SecondsSince1970,Heel_Abs,Heel_Lwd,Lat,LatBow,LatCenter,LatStern,Leg,Line_C,...,interval_duration,mast_brand,gain_forward,gain_lateral,gain_vmg,Line_R2,Line_L2,Line_C2,side_line2,total_line2
57369,2025-06-09T13:37:25.349Z,1.749476e+09,38.5,38.5,43.503292,43.503294,43.503288,43.503282,NaN,84.300,...,52.596,Levi,-0.836055,-3.527920,-1.437218,3.465,6.7,84.300,10.165,94.465
57576,2025-06-09T13:37:46.053Z,1.749476e+09,41.9,41.9,43.505685,43.505687,43.505681,43.505675,NaN,64.900,...,52.596,Levi,-5.698419,-6.453805,0.602422,4.300,6.1,64.900,10.400,75.300
56104,2025-06-09T13:34:29.456Z,1.749476e+09,56.1,56.1,43.506923,43.506921,43.506927,43.506933,NaN,127.443,...,67.090,Levi,2.586595,-5.512053,-2.602855,3.400,4.6,127.443,8.000,135.443
51285,2025-06-09T13:15:40.854Z,1.749475e+09,51.8,51.8,43.508422,43.508420,43.508426,43.508432,NaN,119.700,...,68.692,Levi,0.127233,0.900846,0.629310,5.900,7.9,119.700,13.800,133.500
52656,2025-06-09T13:18:42.566Z,1.749475e+09,53.4,53.4,43.502471,43.502472,43.502467,43.502461,NaN,96.700,...,53.803,Levi,-0.743887,-1.440596,-0.119973,6.200,7.8,96.700,14.000,110.700


In [9]:
t_test(only_karl_first_runs_heavy,only_karl_last_runs_light) #general

print("\nUpwind and downwind for Karl:")
print(f"\nWeight of Karl on the first runs: {only_karl_first_runs_heavy['boat_weight'].mean()}, average SOG: {only_karl_first_runs_heavy['SOG'].mean()}, std SOG: {only_karl_first_runs_heavy['SOG'].std()}")
print(f"Weight of Karl on the last runs: {only_karl_last_runs_light['boat_weight'].mean()}, average SOG: {only_karl_last_runs_light['SOG'].mean()}, std SOG: {only_karl_last_runs_light['SOG'].std()}")

T-statistic: -7.879, p-value: 0.000000000000004
The difference is statistically significant, keeping data split.

Upwind and downwind for Karl:

Weight of Karl on the first runs: 106.97500000000001, average SOG: 23.94727146332986, std SOG: 2.042511601997199
Weight of Karl on the last runs: 100.975, average SOG: 24.320151187904965, std SOG: 1.962379480136006


In [10]:
only_karl_first_runs_heavy_upwind = only_karl_first_runs_heavy[only_karl_first_runs_heavy["TWA"]>0]
only_karl_last_runs_light_upwind = only_karl_last_runs_light[only_karl_last_runs_light["TWA"]>0]

t_test(only_karl_first_runs_heavy_upwind,only_karl_last_runs_light_upwind) #upwind

print("\nUpwind for Karl:")
print(f"Weight of Karl on the first runs: {only_karl_first_runs_heavy_upwind['boat_weight'].mean()}, average SOG: {only_karl_first_runs_heavy_upwind['SOG'].mean()}, std SOG: {only_karl_first_runs_heavy_upwind['SOG'].std()}")
print(f"Weight of Karl on the last runs: {only_karl_last_runs_light_upwind['boat_weight'].mean()}, average SOG: {only_karl_last_runs_light_upwind['SOG'].mean()}, std SOG: {only_karl_last_runs_light_upwind['SOG'].std()}")

T-statistic: -1.138, p-value: 0.255372507601836
The difference is not statistically significant, keeping data combined.

Upwind for Karl:
Weight of Karl on the first runs: 106.97499999999997, average SOG: 22.72190383681399, std SOG: 0.5513928546466119
Weight of Karl on the last runs: 100.97500000000001, average SOG: 22.74386574074074, std SOG: 0.725225703905136


In [11]:
only_karl_first_runs_heavy_downwind = only_karl_first_runs_heavy[only_karl_first_runs_heavy["TWA"] <= 0]
only_karl_last_runs_light_downwind = only_karl_last_runs_light[only_karl_last_runs_light["TWA"] <= 0]
t_test(only_karl_first_runs_heavy_downwind, only_karl_last_runs_light_downwind)  # downwind

print("\nDownwind for Karl:")
print(f"Weight of Karl on the first runs: {only_karl_first_runs_heavy_downwind['boat_weight'].mean()}, average SOG: {only_karl_first_runs_heavy_downwind['SOG'].mean()}, std SOG: {only_karl_first_runs_heavy_downwind['SOG'].std()}")
print(f"Weight of Karl on the last runs: {only_karl_last_runs_light_downwind['boat_weight'].mean()}, average SOG: {only_karl_last_runs_light_downwind['SOG'].mean()}, std SOG: {only_karl_last_runs_light_downwind['SOG'].std()}")


T-statistic: 18.894, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

Downwind for Karl:
Weight of Karl on the first runs: 106.97499999999998, average SOG: 27.031662591687045, std SOG: 0.7814322747955662
Weight of Karl on the last runs: 100.97499999999998, average SOG: 26.32492639842983, std SOG: 0.9483400838098708


## t test Gian heavy vs karl not heavy

In [12]:
only_gian_first_runs_light = data_9juin_first_runs[
    (data_9juin_first_runs["boat_name"] == "Gian Stragiotti") |
    ((data_9juin_first_runs["boat_name"] == "SenseBoard") & 
     (data_9juin_first_runs["opponent_name"] == "Karl Maeder"))
]
only_gian_first_runs_light.sample(5)

,ISODateTimeUTC,SecondsSince1970,Heel_Abs,Heel_Lwd,Lat,LatBow,LatCenter,LatStern,Leg,Line_C,...,interval_duration,mast_brand,gain_forward,gain_lateral,gain_vmg,Line_R2,Line_L2,Line_C2,side_line2,total_line2
44521,2025-06-09T12:38:34.457Z,1.749473e+09,52.3,52.3,43.503492,43.503490,43.503496,43.503502,1.0,5.986,...,68.713,Levi,-0.915361,2.033411,0.901897,5.986,9.5,111.297,15.486,126.783
44188,2025-06-09T12:38:01.163Z,1.749473e+09,56.8,56.8,43.506938,43.506937,43.506943,43.506949,1.0,8.090,...,68.713,Levi,0.736268,-2.862663,-1.685541,8.090,8.9,128.600,16.990,145.590
45795,2025-06-09T12:46:33.075Z,1.749473e+09,57.3,57.3,43.504220,43.504218,43.504224,43.504231,1.0,2.900,...,68.000,Levi,8.043319,4.835334,9.072823,1.900,2.9,97.900,4.800,102.700
44671,2025-06-09T12:38:49.454Z,1.749473e+09,46.7,46.7,43.501961,43.501959,43.501965,43.501971,1.0,5.800,...,68.713,Levi,-4.380225,-4.048793,-5.934871,5.800,9.6,104.400,15.400,119.800
45599,2025-06-09T12:46:13.456Z,1.749473e+09,61.7,61.7,43.506294,43.506293,43.506298,43.506304,1.0,6.700,...,68.000,Levi,1.883462,2.264127,2.937938,6.700,10.3,127.100,17.000,144.100


In [13]:
only_gian_last_runs_heavy = data_9juin_last_runs[
    (data_9juin_last_runs["boat_name"] == "Gian Stragiotti") |
    ((data_9juin_last_runs["boat_name"] == "SenseBoard") & 
     (data_9juin_last_runs["opponent_name"] == "Karl Maeder"))
]
only_gian_last_runs_heavy.sample(5)

,ISODateTimeUTC,SecondsSince1970,Heel_Abs,Heel_Lwd,Lat,LatBow,LatCenter,LatStern,Leg,Line_C,...,interval_duration,mast_brand,gain_forward,gain_lateral,gain_vmg,Line_R2,Line_L2,Line_C2,side_line2,total_line2
52293,2025-06-09T13:16:12.959Z,1.749475e+09,58.8,58.8,43.505254,43.505252,43.505258,43.505265,1.0,7.2,...,68.692,Levi,-0.419627,-3.432566,-2.584841,7.2,9.7,120.800,16.9,137.700
58197,2025-06-09T13:37:55.454Z,1.749476e+09,51.7,51.7,43.507047,43.507049,43.507043,43.507037,NaN,3.1,...,52.596,Levi,-13.679324,-0.233925,10.648370,3.1,5.2,72.500,8.3,80.800
54890,2025-06-09T13:25:34.255Z,1.749476e+09,48.4,48.4,43.502311,43.502310,43.502316,43.502322,NaN,8.8,...,61.205,Levi,12.696955,2.359255,9.476943,8.8,9.3,120.000,18.1,138.100
54552,2025-06-09T13:25:00.456Z,1.749476e+09,50.4,50.4,43.506037,43.506035,43.506041,43.506047,NaN,5.8,...,61.205,Levi,3.237419,0.392505,2.213222,5.8,8.2,105.600,14.0,119.600
50085,2025-06-09T13:06:05.556Z,1.749474e+09,57.1,57.1,43.504134,43.504133,43.504139,43.504145,NaN,9.2,...,61.789,Levi,16.422281,4.916516,13.693555,9.2,11.5,117.813,20.7,138.513


In [14]:
t_test(only_gian_first_runs_light,only_gian_last_runs_heavy) #general

print("\nUpwind and downwind for Gian:")
print(f"\nWeight of Gian on the first runs: {only_gian_first_runs_light['boat_weight'].mean()}, average SOG: {only_gian_first_runs_light['SOG'].mean()}, std SOG: {only_gian_first_runs_light['SOG'].std()}")
print(f"Weight of Gian on the last runs: {only_gian_last_runs_heavy['boat_weight'].mean()}, average SOG: {only_gian_last_runs_heavy['SOG'].mean()}, std SOG: {only_gian_last_runs_heavy['SOG'].std()}")

T-statistic: -11.650, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

Upwind and downwind for Gian:

Weight of Gian on the first runs: 109.08999999999999, average SOG: 24.236319275008714, std SOG: 2.135963842904484
Weight of Gian on the last runs: 115.08999999999999, average SOG: 24.825573344872346, std SOG: 2.1231026500669974


In [15]:
only_gian_first_runs_light_upwind = only_gian_first_runs_light[only_gian_first_runs_light["TWA"]>0]
only_gian_last_runs_heavy_upwind = only_gian_last_runs_heavy[only_gian_last_runs_heavy["TWA"]>0]

t_test(only_gian_first_runs_light_upwind,only_gian_last_runs_heavy_upwind) #upwind

print("\nUpwind for Gian:")
print(f"Weight of Gian on the first runs: {only_gian_first_runs_light_upwind['boat_weight'].mean()}, average SOG: {only_gian_first_runs_light_upwind['SOG'].mean()}, std SOG: {only_gian_first_runs_light_upwind['SOG'].std()}")
print(f"Weight of Gian on the last runs: {only_gian_last_runs_heavy_upwind['boat_weight'].mean()}, average SOG: {only_gian_last_runs_heavy_upwind['SOG'].mean()}, std SOG: {only_gian_last_runs_heavy_upwind['SOG'].std()}")

T-statistic: -8.572, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

Upwind for Gian:
Weight of Gian on the first runs: 109.08999999999997, average SOG: 22.98097323600973, std SOG: 0.7538625153686097
Weight of Gian on the last runs: 115.09000000000002, average SOG: 23.180486862442038, std SOG: 0.8136933474979365


In [16]:
only_gian_first_runs_light_downwind = only_gian_first_runs_light[only_gian_first_runs_light["TWA"] <= 0]
only_gian_last_runs_heavy_downwind = only_gian_last_runs_heavy[only_gian_last_runs_heavy["TWA"] <= 0]
t_test(only_gian_first_runs_light_downwind, only_gian_last_runs_heavy_downwind)  # downwind

print("\nDownwind for Gian:")
print(f"Weight of Gian on the first runs: {only_gian_first_runs_light_downwind['boat_weight'].mean()}, average SOG: {only_gian_first_runs_light_downwind['SOG'].mean()}, std SOG: {only_gian_first_runs_light_downwind['SOG'].std()}")
print(f"Weight of Gian on the last runs: {only_gian_last_runs_heavy_downwind['boat_weight'].mean()}, average SOG: {only_gian_last_runs_heavy_downwind['SOG'].mean()}, std SOG: {only_gian_last_runs_heavy_downwind['SOG'].std()}")


T-statistic: 10.288, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

Downwind for Gian:
Weight of Gian on the first runs: 109.08999999999999, average SOG: 27.405528255528253, std SOG: 0.7868109632056288
Weight of Gian on the last runs: 115.08999999999997, average SOG: 26.918731563421826, std SOG: 1.2548178334314914
